# DermaSense: Full Model Training on Kaggle / Google Colab (GPU)
**Team Healthnautix | Smart India Hackathon 2025 | Problem Statement 25049**

This notebook trains the DermaSense AI skin condition awareness model end-to-end on GPU:
1. Ingests HAM10000 (`kmader/skin-cancer-mnist-ham10000`) and DermNet (`shubhamgoel27/dermnet`)
2. Generates unified labels and performs **GroupStratified 70/15/15 split** by `lesion_id`
3. Trains an ImageNet-pretrained `efficientnet_b3` with Two-Stage transfer learning & Mixed Precision
4. Calibrates probabilities with Temperature Scaling (reports ECE)
5. Calibrates OOD detection with Max-Softmax and Energy Score thresholds
6. Evaluates strictly on the unseen TEST split (Accuracy, Macro-F1, ROC-AUC, Melanoma Sensitivity)
7. Exports to `model.onnx` with numerical equivalence verification
8. Bundles `model.onnx`, `temperature.json`, and `labels.json` for deployment.

In [ ]:
# 1. Install required dependencies
!pip install -q timm onnx onnxruntime scikit-learn seaborn pyyaml

In [ ]:
# 2. Verify GPU compute environment
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU accelerator not detected! In Kaggle, select Settings -> Accelerator -> GPU T4 x2.")

In [ ]:
# 3. Clone or create directory structure
import os
from pathlib import Path

os.makedirs("data/raw/not_skin", exist_ok=True)
os.makedirs("ml/weights", exist_ok=True)
os.makedirs("ml/logs", exist_ok=True)
os.makedirs("docs", exist_ok=True)

In [ ]:
# 4. Generate Not-Skin Reference Class (500 images)
import torchvision.datasets as d
from PIL import Image

fmnist = d.FashionMNIST(root="data/raw/_fmnist", train=True, download=True)
for idx in range(500):
    img, _ = fmnist[idx]
    rgb_img = img.convert("RGB").resize((300, 300), Image.Resampling.BILINEAR)
    rgb_img.save(f"data/raw/not_skin/not_skin_{idx:05d}.png")
print("Saved 500 not_skin reference images.")

In [ ]:
# 5. Check Kaggle input datasets
import glob

ham_candidates = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)
dermnet_candidates = glob.glob("/kaggle/input/**/dermnet*", recursive=True)

print(f"HAM10000 files found: {ham_candidates}")
print(f"DermNet folders found: {dermnet_candidates}")

In [ ]:
# 6. Execute Dataset Preparation & Splitting
# Copy ml/data/prepare.py into Kaggle environment and run
!python ml/data/prepare.py

In [ ]:
# 7. Train Model with Two-Stage Protocol on GPU (Mixed Precision)
!python ml/train.py --config ml/config.yaml

In [ ]:
# 8. Evaluate on Unseen TEST Split
!python ml/evaluate.py --checkpoint ml/weights/best_model.pt --test-csv data/test.csv

In [ ]:
# 9. Export to ONNX & Verify Parity
!python ml/export_onnx.py --checkpoint ml/weights/best_model.pt --output ml/weights/model.onnx

In [ ]:
# 10. Bundle artifacts for local deployment
!zip -j dermasense_model_bundle.zip ml/weights/model.onnx ml/weights/temperature.json ml/weights/labels.json ml/weights/best_model.pt docs/MODEL_CARD.md
print("=" * 80)
print("TRAINING COMPLETE! Download 'dermasense_model_bundle.zip' from Kaggle output.")
print("Extract files directly into: C:\\Users\\WELCOME\\.gemini\\antigravity\\scratch\\dermasense\\ml\\weights\\")
print("=" * 80)